# Lorentz metasurface, staged training on the v3 dataset

**Pipeline (heterogeneous-metasurface slides):** geometry $\mathbf{x}$ (4 numbers per cell: d, l, w, g) $\rightarrow$ Lorentz parameters $\Theta$ (each cell's "bell": strength $\omega_p$, pitch $\omega_0$, decay $\gamma$) $\rightarrow$ fixed physics $f_r$ $\rightarrow$ spectrum. Only $\mathbf{x}\rightarrow\Theta$ is unknown, so that is what the networks learn.

**Concept #1 (what we train here):** the fixed physics block is non-convex and untrainable, which traps gradient descent, so it is *relaxed* into a trainable decoder $D$. Interpretability of $\Theta$ is traded for trainability. The full fixed-physics version is included as an appendix.

**Staged protocol:**
1. **Stage 1, pretrain.** Train $f_{\theta_r}(f_{\theta_1}(x))$, here `decoder(unitary(x))`, to predict **1x1** spectra. Note: the "Lookup Table" label on the slide is a typo (confirmed by Dr. Malof); $f_{\theta_1}$ is an ordinary neural network.
2. **Stage 2, joint.** Warm-start everything and train on **2x2** supercells, now including the interaction network $f_{\theta_2}$.

Every section below gives the equation, the meaning of each variable, and the code that implements it, together.

In [ ]:
import os, numpy as np, torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Mount Drive only on Colab; env vars let you point elsewhere without edits.
if os.path.exists("/content") and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

V3_DIR   = "/content/drive/MyDrive/malof_lab/power_tx_dataset/v3"
NPZ_1X1  = os.environ.get("NB_NPZ_1X1", f"{V3_DIR}/preprocessed_1x1.npz")
NPZ_2X2  = os.environ.get("NB_NPZ_2X2", f"{V3_DIR}/preprocessed_2x2.npz")
CKPT_DIR = os.environ.get("NB_CKPT_DIR", ".")   # point at Drive to persist checkpoints

CHANNELS = 4          # per-cell geometry, v3 feat_names order: d, l, w, g
KERNEL   = 3          # K x K neighbourhood window (odd)
LATENT   = 64         # dim of z_i, the relaxed Lorentz parameters
REL_ENCODING = os.environ.get("NB_REL", "offset")  # none | offset | offset_dist | embed
REL_EMB_DIM  = 8      # size of the learned per-offset embedding (embed mode)
N_POLE   = 8          # conjugate pole pairs per cell (vector-fitting variant)
HIDDEN   = int(os.environ.get("NB_HIDDEN", 2000))  # repo default; WARNING: overfits these
N_HIDDEN = int(os.environ.get("NB_NHID", 10))      # small files. Consider 256 / 4.
BATCH    = 128
EPOCHS   = int(os.environ.get("NB_EPOCHS", 500))
LR, WEIGHT_DECAY = 1e-4, 1e-4
LR_DECAY, LR_PATIENCE, EVAL_STEP, STOP = 0.2, 10, 10, 1e-7
TEST_SPLIT = 0.15

PRETRAIN_CKPT = f"{CKPT_DIR}/concept_one_pretrain.pt"
JOINT_CKPT    = f"{CKPT_DIR}/concept_one_joint_2x2.pt" 

## 1. Data: the v3 `.npz` files

| key | shape | meaning |
|---|---|---|
| `atoms` | (n, n_cells, 4) | per-cell geometry, **cell-major**, cells in row-major order |
| `geom` | (n, n_cells·4) | the same, flattened; plain `reshape(-1, N, N, 4)` is correct |
| `T` | (n, F) | **training target**: power transmission $T=\lvert S_{21}\rvert^2 \in [0,1]$ |
| `S11`, `S21` | (n, F) complex | full scattering parameters, $r$ and $t$ |
| `freq_GHz` | (F,) | the real frequency axis, wired into the physics appendix |

Two details that matter:

* **Unitary 1x1.** Stage 1 wants periodic unit cells. If the 1x1 file stores several identical cells (a unitary 2x2), we verify they match and keep one; anything else raises instead of guessing.
* **Per-channel normalisation.** Statistics are shared across cell positions because the unitary network $U$ is shared: the same (d, l, w, g) must map to the same normalised vector no matter which cell it sits in. Min-max to $[-1, 1]$, fit on the training split only.

In [ ]:
def load_v3(path, grid_n, x_max=None, x_min=None):
    """-> (train_loader, val_loader, test_grid, test_y, freq_GHz, (x_max, x_min))"""
    d = np.load(path, allow_pickle=True)
    atoms = d["atoms"].astype("float32")      # (n, n_cells, 4)
    Y     = d["T"].astype("float32")          # (n, F) target |S21|^2
    freq  = d["freq_GHz"].astype("float64")   # (F,)
    n, n_cells, ch = atoms.shape
    assert ch == CHANNELS, atoms.shape
    if n_cells != grid_n * grid_n:            # unitary file -> collapse to 1 cell
        assert grid_n == 1 and np.allclose(atoms, atoms[:, :1, :], atol=1e-5), \
            f"atoms {atoms.shape} incompatible with grid_n={grid_n}"
        atoms = atoms[:, :1, :]

    a_tr, a_te, y_tr, y_te = train_test_split(atoms, Y, test_size=TEST_SPLIT, random_state=0)

    def norm(a, mx=None, mn=None):            # per CHANNEL, shared across positions
        flat = a.reshape(-1, CHANNELS)
        if mx is None:
            mx, mn = flat.max(0), flat.min(0)
        rng = np.where((mx - mn) / 2 == 0, 1e-8, (mx - mn) / 2)
        out = (flat - (mx + mn) / 2) / rng
        return out.reshape(a.shape).astype("float32"), mx, mn

    a_tr, x_max, x_min = norm(a_tr, x_max, x_min)   # fit on train (or reuse given stats)
    a_te, _, _ = norm(a_te, x_max, x_min)
    g_tr = a_tr.reshape(-1, grid_n, grid_n, CHANNELS)
    g_te = a_te.reshape(-1, grid_n, grid_n, CHANNELS)
    g_tr, g_va, y_tr, y_va = train_test_split(g_tr, y_tr, test_size=0.2, random_state=0)

    class DS(Dataset):
        def __init__(self, x, y): self.x, self.y = torch.tensor(x), torch.tensor(y)
        def __len__(self): return len(self.x)
        def __getitem__(self, i): return self.x[i], self.y[i]

    return (DataLoader(DS(g_tr, y_tr), BATCH, shuffle=True),
            DataLoader(DS(g_va, y_va), BATCH),
            g_te, y_te, freq, (x_max, x_min))

## 2. The physics the model is built around

One Lorentz oscillator, the "bell":

$$\chi(\omega) = \frac{\omega_p^2}{\omega_0^2 - \omega^2 - i\gamma\omega}$$

* $\omega$: probe frequency (the dataset's `freq_GHz`, later normalised by its mean)
* $\omega_p$: strength, how loudly the bell rings (code: `wp_e`, `wp_m`)
* $\omega_0$: resonance, the bell's pitch (code: `w0_e`, `w0_m`)
* $\gamma$: damping, how fast the ring dies (code: `g_e`, `g_m`)
* complex because the charges lag the drive; the imaginary part **is** absorption

Materials sum their bells, and in a supercell **all cells pour into one pool** (the slides' sums to $4N_e$):

$$\epsilon_r(\omega)=\epsilon_\infty+\sum_j \chi_{e,j}(\omega),\qquad \mu_r(\omega)=\mu_\infty+\sum_j \chi_{m,j}(\omega)$$

Then fixed slab formulas, no learnable parameters:

$$n=\sqrt{\epsilon_r\mu_r}\ (\mathrm{Im}\,n\ge 0),\qquad Z=\sqrt{\mu_r/\epsilon_r},\qquad t=\frac{1}{\cos(nk_0d)-\tfrac{i}{2}(Z^{-1}+Z)\sin(nk_0d)}$$

$n$ slows and absorbs the wave, $Z$ mismatch vs vacuum ($Z=1$) reflects it, and the cos/sin terms are the echo interference between the slab's two faces. The target is $T=\lvert t\rvert^2=\lvert S_{21}\rvert^2$, exactly the `T` array. $k_0d$ is the optical thickness; a trainable `kd_scale` absorbs $c$, $d$, and the frequency normalisation.

**Why we relax it:** $f_r$ is fixed and highly non-convex, so it acts like a non-convex loss with bad local minima. Concept #1 therefore swaps $f_r$ for a trainable decoder $D$; the fixed version lives in the appendix.

## 3. The model: Eq. 3 and Eq. 4

Physics ansatz (Eq. 3): a cell's parameters are its isolated value plus **one independent additive nudge per neighbour**,

$$\Theta_i=\tilde\Theta_i+\sum_{k\in S(i)}\delta\tilde\Theta_{i,k}$$

where $S(i)$ is the neighbour index set (here a $K{\times}K$ window with wrap-around padding, matching periodic tiling). Neural version (Eq. 4), with the relaxed latent $z_i$ standing in for $\Theta_i$:

$$z_i=\underbrace{f_{\theta_1}(x_i)}_{\texttt{unitary}\ U}+\sum_{k\in S(i)}\underbrace{f_{\theta_2}(x_i,x_k,\Delta_{ik})}_{\texttt{interaction}\ V},\qquad \hat y=\tfrac{1}{N^2}\sum_i \underbrace{D(z_i)}_{\texttt{decoder}}$$

* $x_i\in\mathbb{R}^4$: the cell's geometry (d, l, w, g), normalised
* $\Delta_{ik}$: the neighbour's relative offset, 2 numbers, so $V$ knows *where* the neighbour sits
* $U, V, D$ are **shared across every cell**, which is what makes the model size-invariant: train on 2x2, run on any $N{\times}N$
* averaging per-cell spectra is the relaxed stand-in for pooling all oscillators into one $\epsilon_r,\mu_r$

The MLP block is the Deng et al. baseline: Linear $\rightarrow$ BatchNorm $\rightarrow$ ReLU hidden layers, bare Linear head.

In [ ]:
class MLP(nn.Module):
    """Linear -> BatchNorm -> ReLU hidden blocks, bare Linear output head."""
    def __init__(self, layers):
        super().__init__()
        mods = []
        for a, b in zip(layers[:-2], layers[1:-1]):
            mods += [nn.Linear(a, b), nn.BatchNorm1d(b), nn.ReLU()]
        mods.append(nn.Linear(layers[-2], layers[-1]))
        self.net = nn.Sequential(*mods)

    def forward(self, x):
        return self.net(x)

## The relative-position ablation: `rel_encoding`

Angle + distance is just the **polar form** of the offset $(\Delta r, \Delta c)$ the interaction network already receives, so it adds no information, only a different encoding. A raw angle also has a wrap discontinuity (179° vs −181°); the safe angular form is $(\cos\theta, \sin\theta) = (\Delta c, \Delta r)/\mathrm{dist}$, which the modes below already span. Four modes, one config switch (`REL_ENCODING`):

* **none**: V is blind to where the neighbour sits
* **offset** *(default, exactly the current model)*: $(\Delta r/\mathrm{pad},\ \Delta c/\mathrm{pad})$
* **offset_dist**: adds $\mathrm{dist}/(\mathrm{pad}\sqrt{2}) \in (0,1]$; only starts to matter at $K \ge 5$ where distances vary
* **embed**: a learned vector per discrete offset slot (transformer-style relative embedding), indexed by $(\Delta r{+}\mathrm{pad})K + (\Delta c{+}\mathrm{pad})$; most expressive, still size-invariant because it depends only on the offset, never on $N$

Run the three-way ablation on the Stage 2 joint fine-tune and compare test MSE.

In [ ]:
import math

class RelEncoding(nn.Module):
    """Builds the <rel features> appended to [x_i, x_k] for each neighbour slot."""
    MODES = ("none", "offset", "offset_dist", "embed")

    def __init__(self, mode="offset", K=3, emb_dim=8):
        super().__init__()
        assert mode in self.MODES, mode
        self.mode, self.K, self.pad = mode, K, K // 2
        self.denom = float(max(self.pad, 1))
        self.diag = self.denom * math.sqrt(2.0)
        self.table = nn.Embedding(K * K, emb_dim) if mode == "embed" else None

    @property
    def extra_dim(self):
        return {"none": 0, "offset": 2, "offset_dist": 3,
                "embed": self.table.embedding_dim if self.table else 0}[self.mode]

    def forward(self, di, dj, B, device, dtype):
        if self.mode == "none":
            return None
        if self.mode == "embed":
            idx = torch.tensor((di + self.pad) * self.K + (dj + self.pad), device=device)
            return self.table(idx).to(dtype).unsqueeze(0).expand(B, -1)
        vals = [di / self.denom, dj / self.denom]
        if self.mode == "offset_dist":
            vals.append(math.sqrt(di * di + dj * dj) / self.diag)
        return torch.tensor(vals, device=device, dtype=dtype).expand(B, len(vals))

In [ ]:
class ConceptOneMetasurface(nn.Module):
    """z_i = U(x_i) + sum_k V(x_i, x_k, rel_ik);  y_hat = mean_i D(z_i).

    N == 1 (or K == 1) short-circuits to D(U(x)): no distinct neighbours exist,
    so this IS the Stage 1 pretraining model f_theta_r(f_theta_1(x)).
    """
    def __init__(self, K=3, C=4, n_freq=2001, latent_dim=64, hidden=256,
                 n_hidden=4, rel_encoding="offset", rel_emb_dim=8):
        super().__init__()
        if K % 2 != 1:
            raise ValueError("K must be odd so each window has a center cell.")
        self.K, self.C, self.pad = K, C, K // 2
        self.n_freq, self.latent_dim = n_freq, latent_dim
        self.rel = RelEncoding(rel_encoding, K, rel_emb_dim)
        pair_in = 2 * C + self.rel.extra_dim
        self.unitary     = MLP([C] + [hidden] * n_hidden + [latent_dim])      # f_theta_1
        self.interaction = MLP([pair_in] + [hidden] * n_hidden + [latent_dim])# f_theta_2
        self.decoder     = MLP([latent_dim] + [hidden] * n_hidden + [n_freq]) # f_theta_r

    def forward(self, grid):
        B, N, _, C = grid.shape
        cells = grid.reshape(B * N * N, C)
        base = self.unitary(cells).reshape(B, N * N, self.latent_dim)

        if N == 1 or self.K == 1:   # Stage 1 path: D(U(x))
            out = self.decoder(base.reshape(B * N * N, self.latent_dim))
            return out.reshape(B, N * N, self.n_freq).mean(dim=1)

        # wrap-around padding: the supercell tiles periodically
        src = (torch.arange(N + 2 * self.pad, device=grid.device) - self.pad) % N
        padded = grid[:, src[:, None], src[None, :], :]

        feats, targets = [], []
        for idx in range(N * N):
            i, j = idx // N, idx % N
            center = grid[:, i, j, :]
            for di in range(-self.pad, self.pad + 1):
                for dj in range(-self.pad, self.pad + 1):
                    if di == 0 and dj == 0:
                        continue
                    neigh = padded[:, i + di + self.pad, j + dj + self.pad, :]
                    pieces = [center, neigh]
                    rel = self.rel(di, dj, B, grid.device, grid.dtype)
                    if rel is not None:
                        pieces.append(rel)
                    feats.append(torch.cat(pieces, dim=1))
                    targets.append(idx)

        delta = self.interaction(torch.cat(feats, dim=0))
        delta = delta.reshape(len(targets), B, self.latent_dim)
        latent = base.clone()
        for k, idx in enumerate(targets):
            latent[:, idx, :] = latent[:, idx, :] + delta[k]

        spectra = self.decoder(latent.reshape(B * N * N, self.latent_dim))
        return spectra.reshape(B, N * N, self.n_freq).mean(dim=1)

In [ ]:
def fit(model, tr, va, ckpt, epochs=EPOCHS):
    """Adam + MSE; save best-val weights to `ckpt`; decay LR on plateau."""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.MSELoss()
    best, bad = float("inf"), 0
    for ep in range(epochs):
        model.train()
        tl = 0.0
        for x, y in tr:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward(); opt.step()
            tl += loss.item() * len(x)
        tl /= len(tr.dataset)
        if (ep + 1) % EVAL_STEP == 0 or ep == epochs - 1:
            model.eval(); vl = 0.0
            with torch.no_grad():
                for x, y in va:
                    vl += crit(model(x.to(device)), y.to(device)).item() * len(x)
            vl /= len(va.dataset)
            print(f"epoch {ep+1:4d}  train {tl:.6f}  val {vl:.6f}")
            if vl < best:
                best, bad = vl, 0
                torch.save(model.state_dict(), ckpt)
            else:
                bad += 1
                if bad >= LR_PATIENCE:
                    for g in opt.param_groups:
                        g["lr"] *= LR_DECAY
                    bad = 0
                    print("  lr ->", opt.param_groups[0]["lr"])
            if vl < STOP:
                break
    return best


def test_mse(model, g_te, y_te):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(g_te).to(device)).cpu().numpy()
    mse = float(((pred - y_te) ** 2).mean())
    print("test MSE:", mse)
    return pred, mse


def plot_spectra(freq, y_true, y_pred, idx=(0, 1)):
    fig, axes = plt.subplots(1, len(idx), figsize=(11, 3), sharey=True)
    for ax, i in zip(np.atleast_1d(axes), idx):
        ax.plot(freq, y_true[i], lw=1, label="simulated")
        ax.plot(freq, y_pred[i], lw=1, label="predicted")
        ax.set_xlabel("frequency (GHz)"); ax.set_ylim(-0.05, 1.05)
    np.atleast_1d(axes)[0].set_ylabel("power transmission T")
    np.atleast_1d(axes)[0].legend()
    plt.tight_layout(); plt.show()

## 4. Stage 1: pretrain $f_{\theta_r}(f_{\theta_1}(x))$ on 1x1

A 1x1 metasurface is a periodic unit cell: no distinct neighbours exist, so the forward pass is literally `decoder(unitary(x))`. The interaction network sits untouched at its random initialisation (it gets no gradients on this path), which is exactly what "pre-train $f_{\theta_1}$ and $f_{\theta_r}$" on the slides means. Both keep updating in Stage 2.

In [ ]:
tr1, va1, gte1, yte1, freq, stats1 = load_v3(NPZ_1X1, grid_n=1)
n_freq = yte1.shape[1]
print("1x1 splits:", len(tr1.dataset), len(va1.dataset), len(gte1), "| n_freq:", n_freq)

model = ConceptOneMetasurface(K=KERNEL, C=CHANNELS, n_freq=n_freq,
                              latent_dim=LATENT, hidden=HIDDEN, n_hidden=N_HIDDEN,
                              rel_encoding=REL_ENCODING, rel_emb_dim=REL_EMB_DIM)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M parameters")

fit(model, tr1, va1, PRETRAIN_CKPT)
model.load_state_dict(torch.load(PRETRAIN_CKPT, map_location=device))
pred1, _ = test_mse(model, gte1, yte1)
plot_spectra(freq, yte1, pred1)

## 5. Stage 2: joint fine-tune on 2x2

Warm-start **all** networks from Stage 1, then train on heterogeneous 2x2 supercells; $f_{\theta_2}$ now receives gradients and learns the neighbour nudges $\delta z$.

Caveat kept consistent with the repo scripts: normalisation statistics are refit on the 2x2 training split. Since both files sample the same geometry ranges the stats land very close to Stage 1's; to force exact reuse, pass `x_max=stats1[0], x_min=stats1[1]` to `load_v3`.

In [ ]:
tr2, va2, gte2, yte2, freq2, stats2 = load_v3(NPZ_2X2, grid_n=2)
assert yte2.shape[1] == n_freq and np.allclose(freq2, freq)
print("2x2 splits:", len(tr2.dataset), len(va2.dataset), len(gte2))

model2 = ConceptOneMetasurface(K=KERNEL, C=CHANNELS, n_freq=n_freq,
                               latent_dim=LATENT, hidden=HIDDEN, n_hidden=N_HIDDEN,
                              rel_encoding=REL_ENCODING, rel_emb_dim=REL_EMB_DIM)
model2.load_state_dict(torch.load(PRETRAIN_CKPT, map_location="cpu"))
print("warm-started every network from Stage 1")

fit(model2, tr2, va2, JOINT_CKPT)
model2.load_state_dict(torch.load(JOINT_CKPT, map_location=device))
pred2, _ = test_mse(model2, gte2, yte2)
plot_spectra(freq, yte2, pred2)

## Variant 2: the vector-fitting decoder (`ConceptOneVF`)

Same Eq. 4 trunk (U + V, additive, with the same `rel_encoding` switch), but the decoder is a **pole-residue rational** in $s = i\omega$ (classical vector fitting):

$$t(s) = \sum_k \left[\frac{r_k}{s - p_k} + \frac{\bar r_k}{s - \bar p_k}\right] + d + h\,s, \qquad T = |t|^2$$

* the head is a **single Linear with no activation**, so Eq. 4's additivity carries exactly into raw parameter space
* stability by construction: $p = -(\mathrm{softplus}(\alpha) + 0.01) + i\beta \Rightarrow \mathrm{Re}\,p < 0$; $\beta$ places the resonance, $\alpha$ its width
* **union pooling**: residues (and $d, h$) are divided / averaged over the $N^2$ cells, so a unitary 2x2 reproduces its 1x1 exactly
* init = physics as a starting point: tiny weights, $\beta$ pre-scattered across the band, $d$-slot bias 1 $\Rightarrow$ the model starts as a transparent slab ($T \approx 1$) and training only deepens and slides resonances

In [ ]:
DELTA = 0.01

def _vf_head(d_in, n_pole):
    head = nn.Linear(d_in, 4 * n_pole + 2)   # [alpha | beta | Re r | Im r | d | h]
    with torch.no_grad():
        head.weight.mul_(0.05)
        head.bias.zero_()
        head.bias[n_pole:2 * n_pole] = torch.linspace(0.2, 2.0, n_pole)  # beta seeds
        head.bias[-2] = 1.0                                              # d: transparent slab
    return head


class ConceptOneVF(nn.Module):
    """Eq. 4 trunk + linear VF head + fixed pole-residue composition."""

    def __init__(self, K=3, C=4, n_freq=2001, latent_dim=64, hidden=256,
                 n_hidden=4, n_pole=8, freqs=None, rel_encoding="offset", rel_emb_dim=8):
        super().__init__()
        assert K % 2 == 1
        self.K, self.pad, self.latent_dim, self.n_pole = K, K // 2, latent_dim, n_pole
        self.rel = RelEncoding(rel_encoding, K, rel_emb_dim)
        pair_in = 2 * C + self.rel.extra_dim
        self.unitary = MLP([C] + [hidden] * n_hidden + [latent_dim])
        self.interaction = MLP([pair_in] + [hidden] * n_hidden + [latent_dim])
        self.vf_head = _vf_head(latent_dim, n_pole)
        w = torch.as_tensor(np.asarray(freqs), dtype=torch.float32)
        self.register_buffer("w", w / w.mean())              # normalised axis, ~1

    def _compose_vf(self, raw, U):
        n = self.n_pole
        a, b = raw[..., 0:n], raw[..., n:2 * n]
        rr, ri = raw[..., 2 * n:3 * n], raw[..., 3 * n:4 * n]
        d, h = raw[..., -2].mean(dim=1), raw[..., -1].mean(dim=1)
        p = torch.complex(-(nn.functional.softplus(a) + DELTA), b)   # Re p < 0: stable
        r = torch.complex(rr, ri) / U                                # unitary identity
        B = raw.shape[0]
        p, r = p.reshape(B, U * n, 1), r.reshape(B, U * n, 1)        # union over cells
        s = torch.complex(torch.zeros_like(self.w), self.w)          # s = i*omega
        t = (r / (s - p) + r.conj() / (s - p.conj())).sum(dim=1)     # Hermitian pair
        t = t + torch.complex(d, torch.zeros_like(d)).unsqueeze(-1) + h.unsqueeze(-1) * s
        return t.real ** 2 + t.imag ** 2

    def forward(self, grid):
        B, N, _, C = grid.shape
        latent = self.unitary(grid.reshape(B * N * N, C)).reshape(B, N * N, self.latent_dim)
        if N > 1 and self.K > 1:
            src = (torch.arange(N + 2 * self.pad, device=grid.device) - self.pad) % N
            padded = grid[:, src[:, None], src[None, :], :]
            feats, targets = [], []
            for idx in range(N * N):
                i, j = idx // N, idx % N
                center = grid[:, i, j, :]
                for di in range(-self.pad, self.pad + 1):
                    for dj in range(-self.pad, self.pad + 1):
                        if di == 0 and dj == 0:
                            continue
                        neigh = padded[:, i + di + self.pad, j + dj + self.pad, :]
                        pieces = [center, neigh]
                        rel = self.rel(di, dj, B, grid.device, grid.dtype)
                        if rel is not None:
                            pieces.append(rel)
                        feats.append(torch.cat(pieces, dim=1))
                        targets.append(idx)
            delta = self.interaction(torch.cat(feats, dim=0))
            delta = delta.reshape(len(targets), B, self.latent_dim)
            for k2, idx in enumerate(targets):
                latent[:, idx, :] = latent[:, idx, :] + delta[k2]
        raw = self.vf_head(latent.reshape(B * N * N, self.latent_dim)).reshape(B, N * N, -1)
        return self._compose_vf(raw, N * N)

In [ ]:
# VF staged run (same protocol; reuses the Stage 1/2 loaders above)
VF_PRE, VF_JOINT = f"{CKPT_DIR}/vf_pretrain.pt", f"{CKPT_DIR}/vf_joint_2x2.pt"

vf1 = ConceptOneVF(K=KERNEL, C=CHANNELS, n_freq=n_freq, latent_dim=LATENT,
                   hidden=HIDDEN, n_hidden=N_HIDDEN, n_pole=N_POLE, freqs=freq,
                   rel_encoding=REL_ENCODING, rel_emb_dim=REL_EMB_DIM)
fit(vf1, tr1, va1, VF_PRE)
vf1.load_state_dict(torch.load(VF_PRE, map_location=device))
pvf1, _ = test_mse(vf1, gte1, yte1)
plot_spectra(freq, yte1, pvf1)

vf2 = ConceptOneVF(K=KERNEL, C=CHANNELS, n_freq=n_freq, latent_dim=LATENT,
                   hidden=HIDDEN, n_hidden=N_HIDDEN, n_pole=N_POLE, freqs=freq,
                   rel_encoding=REL_ENCODING, rel_emb_dim=REL_EMB_DIM)
vf2.load_state_dict(torch.load(VF_PRE, map_location="cpu"))
fit(vf2, tr2, va2, VF_JOINT)
vf2.load_state_dict(torch.load(VF_JOINT, map_location=device))
pvf2, _ = test_mse(vf2, gte2, yte2)
plot_spectra(freq, yte2, pvf2)

## Appendix: the full fixed-physics model

Same $U$ and $V$, but they output raw $\Theta$ (per cell: `n_osc` oscillators $\times$ 6 numbers, `(wp_e, w0_e, g_e, wp_m, w0_m, g_m)`, made positive by softplus) and the decoder is the **fixed** slab physics from Section 2, in differentiable PyTorch. Notes:

* `freqs=freq_GHz` wires the real axis in; it is normalised by its mean so the physics sees a dimensionless $\omega\approx 1$, and the trainable `kd_scale` absorbs $c$, $d$, and that constant. Fix `kd_scale` if the physical thickness is known.
* `reduce="mean"` over cells (the slides write a raw sum to $4N_e$): the mean makes a 2x2 of identical cells reproduce the 1x1 spectrum exactly, which pretraining transfer needs. Confirm the intended convention with Dr. Malof.
* Expect harder optimisation; that difficulty is the whole reason Concept #1 exists.

In [ ]:
import torch.nn.functional as F

class LorentzPhysics(nn.Module):
    """Fixed f_r: raw per-cell Theta (B, M, n_osc, 6) -> power transmission (B, F)."""
    def __init__(self, n_freq, freqs=None, w_min=0.5, w_max=1.5, kd_scale=3.14,
                 train_kd=True, reduce="mean"):
        super().__init__()
        assert reduce in ("mean", "sum")
        self.reduce = reduce
        if freqs is not None:
            w = torch.as_tensor(np.asarray(freqs), dtype=torch.float32)
            assert w.numel() == n_freq
            w = w / w.mean()          # dimensionless, ~1; kd_scale absorbs c*d
        else:
            w = torch.linspace(w_min, w_max, n_freq)
        self.register_buffer("w", w)
        self.eps_inf_raw = nn.Parameter(torch.zeros(1))   # eps_inf = 1 + softplus
        self.mu_inf_raw  = nn.Parameter(torch.zeros(1))
        kd = torch.tensor(float(kd_scale))
        self.kd_scale = nn.Parameter(kd) if train_kd else kd

    def _chi(self, wp, w0, g):        # (B, M, O) -> (B, F) complex susceptibility
        w = self.w
        num = (wp ** 2).unsqueeze(-1)
        den = (w0 ** 2).unsqueeze(-1) - w ** 2 - 1j * g.unsqueeze(-1) * w
        chi = num / den
        return chi.mean(dim=(1, 2)) if self.reduce == "mean" else chi.sum(dim=(1, 2))

    def forward(self, theta):
        p = F.softplus(theta) + 1e-4                      # positivity
        wp_e, w0_e, g_e, wp_m, w0_m, g_m = p.unbind(-1)
        eps = 1.0 + F.softplus(self.eps_inf_raw) + self._chi(wp_e, w0_e, g_e)
        mu  = 1.0 + F.softplus(self.mu_inf_raw)  + self._chi(wp_m, w0_m, g_m)
        n = torch.sqrt(eps * mu)
        n = torch.where(n.imag < 0, -n, n)                # passive branch Im n >= 0
        z = torch.sqrt(mu / eps)
        z = torch.where(z.real < 0, -z, z)
        nkd = n * (self.kd_scale * self.w)                # n * k0 d
        t = 1.0 / (torch.cos(nkd) - 0.5j * (1.0 / z + z) * torch.sin(nkd))
        return (t.abs() ** 2).clamp(max=1.0)


class LorentzMetasurface(nn.Module):
    """U + V predicting Theta, decoded by the FIXED physics block above."""
    def __init__(self, K=3, C=4, n_freq=2001, n_osc=2, hidden=256, n_hidden=4,
                 rel_encoding="offset", rel_emb_dim=8, **phys_kw):
        super().__init__()
        if K % 2 != 1:
            raise ValueError("K must be odd.")
        self.K, self.C, self.pad = K, C, K // 2
        self.n_osc, self.theta_dim = n_osc, n_osc * 6
        self.rel = RelEncoding(rel_encoding, K, rel_emb_dim)
        pair_in = 2 * C + self.rel.extra_dim
        self.unitary     = MLP([C] + [hidden] * n_hidden + [self.theta_dim])
        self.interaction = MLP([pair_in] + [hidden] * n_hidden + [self.theta_dim])
        self.physics     = LorentzPhysics(n_freq, **phys_kw)

    def forward(self, grid):
        B, N, _, C = grid.shape
        theta = self.unitary(grid.reshape(B * N * N, C)).reshape(B, N * N, self.theta_dim)
        if N > 1 and self.K > 1:
            src = (torch.arange(N + 2 * self.pad, device=grid.device) - self.pad) % N
            padded = grid[:, src[:, None], src[None, :], :]
            feats, targets = [], []
            for idx in range(N * N):
                i, j = idx // N, idx % N
                center = grid[:, i, j, :]
                for di in range(-self.pad, self.pad + 1):
                    for dj in range(-self.pad, self.pad + 1):
                        if di == 0 and dj == 0:
                            continue
                        neigh = padded[:, i + di + self.pad, j + dj + self.pad, :]
                        pieces = [center, neigh]
                        rel = self.rel(di, dj, B, grid.device, grid.dtype)
                        if rel is not None:
                            pieces.append(rel)
                        feats.append(torch.cat(pieces, dim=1))
                        targets.append(idx)
            delta = self.interaction(torch.cat(feats, dim=0))
            delta = delta.reshape(len(targets), B, self.theta_dim)
            for k, idx in enumerate(targets):
                theta[:, idx, :] = theta[:, idx, :] + delta[k]
        return self.physics(theta.reshape(B, N * N, self.n_osc, 6))

In [ ]:
# Forward demo with the real frequency axis; train it exactly like Stages 1 and 2.
phys_model = LorentzMetasurface(K=KERNEL, C=CHANNELS, n_freq=n_freq, n_osc=2,
                                hidden=HIDDEN, n_hidden=N_HIDDEN, freqs=freq).eval()
with torch.no_grad():
    demo = phys_model(torch.tensor(gte2[:4]))
print("full-physics forward:", tuple(demo.shape),
      "| T range", round(float(demo.min()), 3), "-", round(float(demo.max()), 3),
      "| normalised w axis", round(float(phys_model.physics.w.min()), 3),
      "-", round(float(phys_model.physics.w.max()), 3))

In [ ]:
# Sanity checks
c = ConceptOneMetasurface(K=3, C=4, n_freq=n_freq, latent_dim=8, hidden=32, n_hidden=2).eval()
g = torch.randn(3, 1, 1, 4)
with torch.no_grad():
    assert torch.allclose(c(g), c.decoder(c.unitary(g.reshape(3, 4))), atol=1e-6)
print("Stage 1 path check: model(1x1) == f_theta_r(f_theta_1(x))  OK")
assert torch.isfinite(demo).all() and float(demo.min()) >= 0.0
print("fixed-physics output finite and in [0, 1]  OK")

vfc = ConceptOneVF(K=3, C=4, n_freq=n_freq, latent_dim=8, hidden=32, n_hidden=2,
                   n_pole=4, freqs=freq).eval()
lastL = [m for m in vfc.interaction.modules() if isinstance(m, nn.Linear)][-1]
lastL.weight.data.zero_(); lastL.bias.data.zero_()
gc = torch.randn(1, 1, 1, 4)
with torch.no_grad():
    assert (vfc(gc) - vfc(gc.expand(1, 2, 2, 4).contiguous())).abs().max() < 1e-4
print("VF unitary identity (2x2 identical == 1x1)  OK")